In [0]:
# Databricks Notebook: bronze_ingest.py
import requests
import json
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col

spark = SparkSession.builder.getOrCreate()

# API endpoint
url = "https://services2.arcgis.com/5I7u4SJE1vUr79JC/arcgis/rest/services/UniversityChapters_Public/FeatureServer/0/query"
params = {
    "where": "State IN ('CA','OR','WA')",
    "outFields": "*",
    "returnGeometry": "true",
    "f": "json"
}

response = requests.get(url, params=params)
data = response.json()

# Generate run_id for this ingest
run_id = datetime.utcnow().strftime("%Y%m%d%H%M%S")

# Persist run_id as a Databricks widget so downstream notebooks can access it
dbutils.widgets.text("run_id", run_id)

bronze_path = f"/Volumes/azure-medallion-university-chapters/bronze/university_chapters/{run_id}/"

# Save raw JSON payload to Bronze
df_json_str = spark.createDataFrame([(json.dumps(data),)], ["json_str"])
schema_str = df_json_str.selectExpr("schema_of_json_agg(json_str)").collect()[0][0]
bronze_df = df_json_str.select(from_json(col("json_str"), schema_str).alias("parsed")).select("parsed.*")

bronze_df.write.mode("overwrite").json(bronze_path)

print(f"Bronze ingest complete. Run ID: {run_id}, Path: {bronze_path}")
